## Building Unit Cleaning Code, for San Diego County

Used for joining with SDGE utility

In [1]:
# import necessary libraries
import pandas as pd
from shapely.geometry import box
import numpy as np
import geopandas as gpd
import os
import matplotlib.pyplot as plt
import fiona
import pandas as pd

# statistical libraries
#import sys
#!{sys.executable} -m pip install statsmodels
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression

In [2]:
# set option to see all data frame columns
pd.set_option('display.max_columns', None)

In [3]:
# load parcels 
parcels = gpd.read_parquet("/../../capstone/electrigrid/data/parcels/parcels_final.parquet").to_crs(epsg=4326)

# filter for San Diego
parcels = parcels[parcels['County'] == "San Diego"]

In [4]:
# import Zillow data 
zillow = gpd.read_parquet("/../../capstone/electrigrid/data/buildings/zillow.parquet").to_crs(epsg=4326)

Building Footprint

Access: https://sat-io.earthengine.app/view/gba

In [5]:
# import building footprint as geopandas dataframe (may take 1-5 minutes)
building = gpd.read_parquet("../../../../../capstone/electrigrid/data/buildings/buildings_ca.parquet").to_crs(epsg=4326)

In [6]:
# load in SDG&E boundary map
sdge_shape = gpd.read_file("/../../capstone/electrigrid/data/utilities/IOU_shapefiles.geojson")

sdge_shape = sdge_shape[sdge_shape['Acronym'] == "SDG&E"]

## Unit-regression for multi-family homes

### Select parcels for multi-family homes

In [16]:
# only keep the geometry and parcel number of parcels
parcels = parcels[['PARNO', 'geometry']]

## select only multi-family data from Zillow
zillow_multi = zillow[zillow['type'] == "Multi"]
zillow_multi = zillow_multi[zillow_multi['code'] != "RR106"]

## crop only to residential parcels
# keep the indices where multi-family homes match to parcels
valid_parcels = parcels.sjoin(zillow_multi, predicate="contains").index.unique()

# select the parcels that match these indices
parcels_res = parcels.loc[valid_parcels]

# confirm that joining with Zillow decreases the number of parcels
assert len(parcels_res) < len(parcels)

# crop to residential buildings (by keeping only those within residential parcels)
valid_buildings = building.sjoin(parcels_res, predicate="intersects").index.unique()
buildings_res = building.loc[valid_buildings]

## join parcels to buildings (keeping observations as parcels, but with building attributes)
# sum number of units per parcel
units_sum = parcels_res.sjoin(zillow_multi, predicate="intersects").groupby(level=0)["unit"].sum()

# join on parcels with summed number of units
parcels_res = parcels_res.join(units_sum)

# confirm that joining with Zillow decreased the number of buildings
assert len(buildings_res) < len(building)

# keep all residential buildings, and add zillow points only where they match up
building_zillow = gpd.sjoin(
    buildings_res,
    zillow_multi,
    predicate = "intersects")

building_zillow = building_zillow.drop(columns = "index_right")

# add in the parcel number to add all of the building volumes for the regression
building_zillow_parcels = building_zillow.sjoin(
                                parcels_res,
                                how = "left",
                                predicate = "intersects")

## FOR BUILDINGS INTERSECTING MORE THAN ONE PARCEL CALCULATE WHICH PARCEL IT INTERSECTS MORE
# change the crs to a projected crs
building_zillow_parcels = building_zillow_parcels.to_crs("EPSG:6933")
parcels_res = parcels_res.to_crs("EPSG:6933")

# calculate intersection area for each building-parcel pair
building_zillow_parcels["intersection_area"] = (
    building_zillow_parcels.geometry.values
    .intersection(parcels_res.loc[building_zillow_parcels["index_right"]].geometry.values).area)

# keep only the parcel with the largest overlap per building
building_zillow_parcels = (
    building_zillow_parcels
    .sort_values("intersection_area", ascending=False)
    .loc[~building_zillow_parcels.index.duplicated(keep="first")]
    .drop(columns="intersection_area"))

# drop the unnecessary columns
building_zillow_parcels = building_zillow_parcels.drop(columns = ['index_right', 'unit_left'])
building_zillow_parcels = building_zillow_parcels.rename(columns = {'unit_right': 'total_unit'})

# change the crs back to grographic
building_zillow_parcels = building_zillow_parcels.to_crs(epsg=4326)
parcels_res = parcels_res.to_crs(epsg=4326)

# reproject data frame to crs with meters as units
building_m = building_zillow_parcels.to_crs("EPSG:6933")

# create column from polygon area
building_m['area_m2'] = building_m.geometry.area

# rename height column to be clear about units
building_m.rename(columns={"height":"height_m"}, inplace = True)

# create volume column
building_m['volume_m3'] = building_m['area_m2'] * building_m['height_m']

# keep only observations with unit data
building_w_units = building_m[~building_m['total_unit'].isna()]

assert building_w_units['total_unit'].isna().sum() == 0

# drop multi-family homes where the total_unit is equal to zero
building_w_units = building_w_units.drop(building_w_units[building_w_units['total_unit'] == 0].index)

# aggregate the volumes by parcel
total_volume_m3 = building_w_units.groupby('PARNO')['volume_m3'].sum(min_count = 1)

# change the series to a dataframe
total_volume_m3 = pd.DataFrame(total_volume_m3)

# rename the column to total_volume_m3
total_volume_m3 = total_volume_m3.rename(columns = {'volume_m3' : 'total_volume_m3'})

# add the total volume to the buildings dataframe
building_w_units = building_w_units.join(total_volume_m3, on = 'PARNO')

# some of the homes don't have a parcel number (so replace the total volume with volume if its na
building_w_units['total_volume_m3'] = building_w_units['total_volume_m3'].fillna(building_w_units['volume_m3'])

# remove duplicates from the parcel number so that it doesn't skew the linear regression
building_w_units = building_w_units.drop_duplicates(subset = 'PARNO', keep = 'first')

# run model
results = smf.ols('total_unit ~ total_volume_m3', data=building_w_units).fit()

# add residuals as a column
building_w_units['residual'] = results.resid.copy()

# keep only observations that are less/equal to 2 standard deviations from residuals
building_units_clean = building_w_units[building_w_units['residual'].abs() <= 2 * building_w_units['residual'].std()]

# save outliers, as we will re-predict them using the regression
building_outliers = building_w_units[building_w_units['residual'].abs() > 2 * building_w_units['residual'].std()]

# rerun linear regression
results_clean = smf.ols('total_unit ~ total_volume_m3', data=building_units_clean).fit()

# save variables
intercept = results_clean.params[0]
slope = results_clean.params[1]

# extract just the multi-family homes where unit info is missing
missing_units = building_m[(building_m['total_unit'].isna()) & (building_m['total_unit'] == 0)]


# aggregate the volumes for the unit regression by parcel
missing_total_volume_m3 = missing_units.groupby('PARNO')['volume_m3'].sum(min_count = 1)

# change the series to a dataframe
missing_total_volume_m3 = pd.DataFrame(missing_total_volume_m3)

# rename the column to missing_total_volume_m3
missing_total_volume_m3 = missing_total_volume_m3.rename(columns = {'volume_m3' : 'missing_total_volume_m3'})

# add the missing total volume to the buildings dataframe
missing_units = missing_units.join(missing_total_volume_m3, on = 'PARNO')

# combine dataframes with missing unit data as well as outliers (since both will be predicted)
missing_outlier_units = pd.concat([building_outliers, missing_units])

assert len(missing_units) < len(missing_outlier_units)

# replace anywhere the missing_total_volume is missing (fill in for the outliers since total_volume was computed before)
missing_outlier_units['missing_total_volume_m3'] = missing_outlier_units['missing_total_volume_m3'].fillna(missing_outlier_units['total_volume_m3'])

# replace unit column with prediction
missing_outlier_units_pred = missing_outlier_units.copy().drop('total_unit', axis = 1)

missing_outlier_units_pred = missing_outlier_units_pred.reset_index(drop=True)

missing_outlier_units_pred['total_unit'] = round(intercept + missing_outlier_units_pred['missing_total_volume_m3'] * slope)

## multiple buildings in one parcel have duplicated units since they were computed on total volume
#  only keep the first observation per parcel
missing_outlier_units_pred = missing_outlier_units_pred.drop_duplicates(subset = 'PARNO', keep = 'first')

# combine multi-family homes data frames
multi_complete = pd.concat([building_units_clean, missing_outlier_units_pred]).to_crs(zillow.crs)

# fill the total_volume for those with the missing_total_volume_m3 
multi_complete['missing_total_volume_m3'] = multi_complete['missing_total_volume_m3'].fillna(multi_complete['total_volume_m3'])

# drop unnecessary columns 
multi_complete = multi_complete.drop(['residual', 'geometry', 'total_volume_m3'], axis = 1)

# rename to total_volume_m3
multi_complete = multi_complete.rename(columns = {'missing_total_volume_m3' : 'total_volume_m3'})


# drop the unit data from parcel res as the new unit data was computed
parcels_res = parcels_res.drop(columns = ['unit'])

# join the parcel data on PARNO to get the parcel geometry
multi_by_parcel = parcels_res.merge(multi_complete, on = 'PARNO')


# set the geometry to the parcel geometry
multi_by_parcel = multi_by_parcel.set_geometry('geometry')

# change the crs to projected 
multi_by_parcel = multi_by_parcel.to_crs("EPSG:6933")

## IN ORDER TO CONCAT THE MULTI-FAMILY HOMES WITH SINGLE FAMILY HOMES THEY MUST ALL BE POINTS
# create centroids for each multi residential parcel 
multi_by_parcel['centroids'] = multi_by_parcel.geometry.centroid

# drop the geometry of multi_by_parcel so the centroid becomes the geometry
multi_by_parcel = multi_by_parcel.drop(columns = 'geometry')

# rename the centroid to geometry
multi_by_parcel = multi_by_parcel.rename(columns = {'centroids' : 'geometry'})

# set the centroid to the geometry
multi_by_parcel_processed = multi_by_parcel.set_geometry('geometry')

# change the CRS back to the Zillow CRS 
multi_by_parcel_processed = multi_by_parcel_processed.to_crs(zillow.crs)

## HANDLING SINGLE FAMILY HOMES
# subset for single family zillow and condos
single_zillow = zillow[zillow['type'] == "Single"].to_crs(zillow.crs)

# keep only single family homes within San Diego County
single_zillow = single_zillow.clip(sdge_shape)

# change all single_zillow to unit = 1
single_zillow['total_unit'] = 1

# drop the unit column
single_zillow = single_zillow.drop(['unit'], axis = 1)

# concat the single and multifamily dataframes
complete_zillow = pd.concat([multi_by_parcel_processed, single_zillow], axis = 0)

/tmp/ipykernel_2636459/3638382881.py:125: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept = results_clean.params[0]
/tmp/ipykernel_2636459/3638382881.py:126: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  slope = results_clean.params[1]


In [17]:
complete_zillow.head()

,PARNO,source,id,height_m,var,region,bbox,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,total_unit,area_m2,volume_m3,total_volume_m3,geometry
0,4236921300,osm,1308122121,3.074826,0.141122,USA,"{'xmax': -117.25140540000001, 'xmin': -117.251...",Multi,1975.0,4.0,None,None,None,1026707.0,None,NaN,7996232,06073007602,h567,SDGE,RI101,2.0,140.235708,431.200368,431.200368,POINT (-117.25142 32.76728)
1,4245420100,ms,UnitedStates_023013221_369659,7.822832,2.910877,USA,"{'xmax': -117.23303161924132, 'xmin': -117.233...",Multi,NaN,NaN,None,None,None,158049.0,None,NaN,7998882,06073007702,h530,SDGE,RI110,4.0,224.418227,1755.586108,1755.586108,POINT (-117.23320 32.79144)
2,4245420700,ms,UnitedStates_023013221_13570,6.449017,3.254450,USA,"{'xmax': -117.23293286671338, 'xmin': -117.233...",Multi,1962.0,4.0,None,None,O,576122.0,living,2212.0,7998886,06073007702,h530,SDGE,RI101,2.0,113.884017,734.439912,734.439912,POINT (-117.23297 32.79055)
3,5712820100,ms,UnitedStates_023013221_591587,3.844132,0.142668,USA,"{'xmax': -117.08611408293011, 'xmin': -117.086...",Multi,1952.0,3.0,None,None,O,479668.0,living,1224.0,8230580,06073012600,h564,SDGE,RI101,2.0,233.630791,898.107588,898.107588,POINT (-117.08621 32.62017)
4,5681511000,osm,956699452,7.273522,0.659870,USA,"{'xmax': -117.08155640000001, 'xmin': -117.081...",Multi,1968.0,6.0,None,None,None,1150682.0,living,3290.0,8226851,06073012302,h577,SDGE,RI104,6.0,139.970610,1018.079361,1018.079361,POINT (-117.08168 32.64212)


In [18]:
complete_zillow[complete_zillow[total_volume_m3].isna()]

,PARNO,source,id,height_m,var,region,bbox,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,total_unit,area_m2,volume_m3,total_volume_m3,geometry
0,4236921300,osm,1308122121,3.074826,0.141122,USA,"{'xmax': -117.25140540000001, 'xmin': -117.251...",Multi,1975.0,4.0,None,None,None,1026707.0,None,NaN,7996232,06073007602,h567,SDGE,RI101,2.0,140.235708,431.200368,431.200368,POINT (-117.25142 32.76728)
1,4245420100,ms,UnitedStates_023013221_369659,7.822832,2.910877,USA,"{'xmax': -117.23303161924132, 'xmin': -117.233...",Multi,NaN,NaN,None,None,None,158049.0,None,NaN,7998882,06073007702,h530,SDGE,RI110,4.0,224.418227,1755.586108,1755.586108,POINT (-117.23320 32.79144)
2,4245420700,ms,UnitedStates_023013221_13570,6.449017,3.254450,USA,"{'xmax': -117.23293286671338, 'xmin': -117.233...",Multi,1962.0,4.0,None,None,O,576122.0,living,2212.0,7998886,06073007702,h530,SDGE,RI101,2.0,113.884017,734.439912,734.439912,POINT (-117.23297 32.79055)
3,5712820100,ms,UnitedStates_023013221_591587,3.844132,0.142668,USA,"{'xmax': -117.08611408293011, 'xmin': -117.086...",Multi,1952.0,3.0,None,None,O,479668.0,living,1224.0,8230580,06073012600,h564,SDGE,RI101,2.0,233.630791,898.107588,898.107588,POINT (-117.08621 32.62017)
4,5681511000,osm,956699452,7.273522,0.659870,USA,"{'xmax': -117.08155640000001, 'xmin': -117.081...",Multi,1968.0,6.0,None,None,None,1150682.0,living,3290.0,8226851,06073012302,h577,SDGE,RI104,6.0,139.970610,1018.079361,1018.079361,POINT (-117.08168 32.64212)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7116844,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Single,NaN,4.0,None,None,O,492864.0,living,2736.0,5134299,06059032065,h542,SDGE,RR101,1.0,NaN,NaN,NaN,POINT (-117.62087 33.59331)
7115882,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Single,NaN,4.0,None,None,O,2444940.0,living,4443.0,5132939,06059032044,h542,SDGE,RR101,1.0,NaN,NaN,NaN,POINT (-117.59029 33.59327)
7115877,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Single,NaN,NaN,None,None,None,2823360.0,None,NaN,5132915,06059032044,h542,SDGE,RR101,1.0,NaN,NaN,NaN,POINT (-117.58842 33.59335)
7115275,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Single,NaN,4.0,None,None,O,948196.0,living,3343.0,5132212,06059032046,h542,SDGE,RR101,1.0,NaN,NaN,NaN,POINT (-117.58295 33.59297)


In [19]:
multi_by_parcel_processed[multi_by_parcel_processed['total_volume_m3'].isna()]

,PARNO,source,id,height_m,var,region,bbox,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,total_unit,area_m2,volume_m3,total_volume_m3,geometry


## Renaming and saving

In [59]:
# save the complete Zillow points
complete_zillow_sd = complete_zillow.copy()

In [ ]:
# takes a really long time :,(
complete_zillow_sd.to_file("data/complete_zillow_sd.geojson", driver='GeoJSON')

KeyboardInterrupt: 